# The Archive: render film shots on Colab

Renders shots of the portfolio film on a Colab GPU, alongside the H200 and the laptop. Each shot
is one self-contained `.blend` file (scene, animation and textures all inside it), made with
`blender/out/colab/` on the laptop.

**Before running:** put the shot files where this notebook can see them, in either place:
- **Google Drive** (best: frames survive a disconnect): upload `approach.blend` and `door.blend`
  from `C:\Users\vedan\Downloads\website\blender\out\colab\` to a Drive folder named
  **`archive-colab`** (so `MyDrive/archive-colab/approach.blend`).
- **This session's disk:** upload them into `/content/archive-colab/` (they are lost when the
  session ends, so download the frames zip at the end).

Then set the runtime to a GPU (Runtime → Change runtime type → A100, L4 or T4) and **Run all**.
It can be stopped and run again at any time: frames already rendered are skipped.

In [ ]:
# Which GPU this session has.
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Settings: the shots, and the render settings the H200 and laptop use for the same film.
SHOTS = ['approach', 'door']
RES = (1920, 1080)
SAMPLES = 192
THRESHOLD = 0.02     # adaptive sampling noise threshold
VOLSTEP = 1.5        # volume step rate (the street haze)

In [ ]:
# Where the shots are, and where the frames go: Google Drive if it can be mounted, else /content.
import os
ROOT = '/content/archive-colab'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    if os.path.isdir('/content/drive/MyDrive/archive-colab'):
        ROOT = '/content/drive/MyDrive/archive-colab'
except Exception as e:
    print('No Drive here (', e, '); using', ROOT)
os.makedirs(ROOT, exist_ok=True)
FRAMES = os.path.join(ROOT, 'frames')
os.makedirs(FRAMES, exist_ok=True)
missing = [s for s in SHOTS if not os.path.exists(os.path.join(ROOT, f'{s}.blend'))]
print('Shots from', ROOT, '| frames to', FRAMES)
print('Missing:', missing or 'none')
assert not missing, 'Upload the .blend files first (see the top of the notebook).'

In [ ]:
# Blender 5.2.1 (the same build as the H200 and the laptop), and the libraries it needs.
import os, subprocess
if not os.path.exists('/content/blender/blender'):
    !apt-get -qq update && apt-get -qq install -y libxi6 libxxf86vm1 libxfixes3 libxrender1 libgl1 libxkbcommon0 libsm6 libegl1 > /dev/null
    !wget -q https://download.blender.org/release/Blender5.2/blender-5.2.1-linux-x64.tar.xz -O /content/blender.tar.xz
    !mkdir -p /content/blender && tar -xf /content/blender.tar.xz -C /content/blender --strip-components=1 && rm /content/blender.tar.xz
!/content/blender/blender --version | head -1

In [ ]:
# The render script Blender runs: open a shot, use the GPU (OptiX, else CUDA), apply the settings,
# render every frame not already on disk.
SCRIPT = r'''
import bpy, os, sys, time
a = sys.argv[sys.argv.index('--') + 1:]
out, w, h, samples, threshold, volstep = a[0], int(a[1]), int(a[2]), int(a[3]), float(a[4]), float(a[5])
k, every = (int(v) for v in a[6].split('/'))
sc = bpy.context.scene
prefs = bpy.context.preferences.addons['cycles'].preferences
for kind in ('OPTIX', 'CUDA'):
    try:
        prefs.compute_device_type = kind
        prefs.get_devices()
        devs = [d for d in prefs.devices if d.type == kind]
        if devs:
            for d in prefs.devices:
                d.use = d.type == kind
            break
    except TypeError:
        continue
sc.cycles.device = 'GPU'
print('[colab] device', kind, [d.name for d in prefs.devices if d.use], flush=True)
sc.render.resolution_x, sc.render.resolution_y = w, h
sc.render.resolution_percentage = 100
sc.cycles.samples = samples
sc.cycles.adaptive_threshold = threshold
sc.cycles.volume_step_rate = volstep
sc.render.image_settings.file_format = 'JPEG'
sc.render.image_settings.quality = 95
for ob in sc.objects:  # the footstep dust and the motes are left out of this film
    if ob.name.startswith(('footdust', 'motes')):
        ob.hide_render = True
os.makedirs(out, exist_ok=True)
for f in range(sc.frame_start - 1, sc.frame_end):
    path = os.path.join(out, f'{f:04d}.jpg')
    if f % every != k or (os.path.exists(path) and os.path.getsize(path) > 0):
        continue
    sc.frame_set(f + 1)
    sc.render.filepath = path
    t = time.time()
    bpy.ops.render.render(write_still=True)
    print(f'[colab] {os.path.basename(out)} {f + 1}/{sc.frame_end} {time.time() - t:.1f}s', flush=True)
print('[colab] done', flush=True)
'''
open('/content/render.py', 'w').write(SCRIPT)
print('ok')

In [ ]:
# Render. Progress prints a line per frame; stop and re-run whenever (finished frames are kept).
import subprocess, sys
for shot in SHOTS:
    cmd = ['/content/blender/blender', '-b', os.path.join(ROOT, f'{shot}.blend'), '-P', '/content/render.py', '--',
           os.path.join(FRAMES, shot), str(RES[0]), str(RES[1]), str(SAMPLES), str(THRESHOLD), str(VOLSTEP), '0/1']
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if line.startswith('[colab]') or 'Error' in line or 'Traceback' in line:
            print(line, end='', flush=True)
    p.wait()
    print(shot, 'exit', p.returncode)

In [ ]:
# How many frames each shot has now.
for shot in SHOTS:
    d = os.path.join(FRAMES, shot)
    n = len([f for f in os.listdir(d) if f.endswith('.jpg')]) if os.path.isdir(d) else 0
    print(shot, n, 'frames')

In [ ]:
# If the frames are NOT on Drive: zip them to download (Files panel, or the link printed below).
import shutil
if not ROOT.startswith('/content/drive'):
    z = shutil.make_archive('/content/archive-colab-frames', 'zip', FRAMES)
    print('Download:', z)
else:
    print('The frames are in Drive:', FRAMES)